# NACC_DemRKL
Copied from the original project. Paths are relative to the main folder.
Original DemRKG generation and training settings are preserved.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'NACC'
METHOD = 'DemRKL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

RAW_KG_PATH = str(raw_primekg_path())

def precise_search_missed_nodes():
    print("==================================================")
    print("🎯 启动未命中节点精准检索...")
    
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    except FileNotFoundError:
        print("❌ 找不到 kg.csv，请检查路径。")
        return

    target_types = ['disease', 'effect/phenotype']
    search_df = df[df['x_type'].isin(target_types)]
    
    missed_terms = {
        "房颤 (AFIB)": "atrial fibrillation",
        "短暂性脑缺血 (TIA)": "transient ischemic attack",
        "脑外伤 (TBI)": "traumatic brain",
        "高胆固醇 (HYPERCHO)": "hypercholesterolemia",
        "糖尿病 (DIABETES)": "diabetes"
    }
    
    for name, keyword in missed_terms.items():
        print(f"\n▶ 正在精准检索【{name}】...")
        
        mask = search_df['x_name'].str.contains(keyword, case=False, na=False)
        results = search_df[mask]['x_name'].unique()
        
        if len(results) > 0:
            print(f"   ✅ 找到 {len(results)} 个相关节点。最匹配的前 10 个如下：")
            sorted_results = sorted(results, key=len)
            for i, res in enumerate(sorted_results[:10]):
                print(f"      [{i+1}] {res}")
        else:
            print(f"   ❌ 完全没有找到包含 '{keyword}' 的节点。")

    print("\n==================================================")
    print("💡 检索完毕。请复制最准确的名称并替换到 Step 2 中。")
    print("==================================================")

if __name__ == "__main__":
    precise_search_missed_nodes()